# SkillBridge AI — Ultimate ML Model v13.0

**AI-Based Job & Skill Recommendation System (SDG-8)**

> *Target: Young Bangladeshi Students & Fresh Graduates*

---

### ML Models Implemented
| Model | Reference |
|---|---|
| TF-IDF + Cosine Similarity | Ajjam & Al-Raweshidy, 2026 |
| Word2Vec Embeddings | Alsaif et al., 2022 |
| Jaccard Coefficient | Alsaif et al., 2022 |
| K-Nearest Neighbors | Research Proposal |
| GRU Career Tracking | Huang, 2022 |
| Linear Regression (Salary) | Standard |
| 6-Stage Recruitment Engine | Chen, 2022 |
| Baseline Keyword Model | Comparison |

### Evaluation Metrics
- Precision@K, Recall@K, F1-Score, Accuracy
- AUC, R², MSE, MAE
- Wilcoxon Test, Cohen's D
- 5-Fold Cross-Validation

### Expected CSV Datasets
1. `JobsFE.csv` — 8 columns, ~10,000 entries
2. `career_guidance_dataset.csv` — 22 columns, ~1,000 entries
3. `job_recommendation_dataset.csv` — 7 columns, ~50,000 entries

---
** SDG-8: Decent Work and Economic Growth**

## Step 1: Install Dependencies

Run this cell first. It auto-installs all required packages and downloads NLTK data.

In [1]:
print("=" * 80)
print(" INSTALLING REQUIRED PACKAGES...")
print("=" * 80)

import subprocess
import sys

def install(package):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
        return True
    except:
        return False

packages = [
    "pandas", "numpy", "scikit-learn", "scipy",
    "nltk", "gensim", "torch", "sentence-transformers",
    "plotly", "matplotlib", "seaborn", "wordcloud",
    "pdfplumber", "python-docx", "openpyxl", "pillow"
]

for pkg in packages:
    result = install(pkg)
    print(f"  {'[OK]' if result else ''} {pkg}")

import nltk
for resource in ['punkt', 'stopwords', 'wordnet', 'vader_lexicon', 'punkt_tab']:
    try:
        nltk.download(resource, quiet=True)
    except:
        pass
print("  [OK] NLTK data")

print("\n[Done] All dependencies installed!")

📦 INSTALLING REQUIRED PACKAGES...
  ✓ pandas
  ✓ numpy
  ✓ scikit-learn
  ✓ scipy
  ✓ nltk
  ✓ gensim
  ✓ torch
  ✓ sentence-transformers
  ✓ plotly
  ✓ matplotlib
  ✓ seaborn
  ✓ wordcloud
  ✓ pdfplumber
  ✓ python-docx
  ✓ openpyxl
  ✓ pillow
  ✓ NLTK data

✅ All dependencies installed!


## Step 2: Imports & Configuration

All imports and system-wide configuration constants.

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
import io
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from datetime import datetime
from collections import Counter, defaultdict
import json
import math

# Google Colab detection
try:
    from google.colab import files
    IN_COLAB = True
    print("[Done] Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("ℹ️  Running locally (not in Colab)")

# Machine Learning
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    roc_auc_score, mean_squared_error, mean_absolute_error, r2_score
)

# Statistical Tests
from scipy.stats import wilcoxon

# NLP
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer

# Word2Vec
from gensim.models import Word2Vec

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# CV Parsing
import pdfplumber
try:
    import docx
    DOCX_AVAILABLE = True
except:
    DOCX_AVAILABLE = False

import torch

warnings.filterwarnings('ignore')

print(f"[Done] All imports successful!")
print(f"️  GPU Available: {torch.cuda.is_available()}")
print(f" DOCX Support: {DOCX_AVAILABLE}")

✅ Running in Google Colab
✅ All imports successful!
🖥️  GPU Available: True
📦 DOCX Support: True


In [4]:
# SYSTEM CONFIGURATION

class Config:
    """Complete system configuration — edit values here to tune the system."""

    # Dataset file names
    DATASETS = {
        'jobs':      'JobsFE.csv',
        'career':    'career_guidance_dataset.csv',
        'recommend': 'job_recommendation_dataset.csv'
    }

    # TF-IDF domain-specific keyword weights (Ajjam & Al-Raweshidy, 2026)
    DOMAIN_KEYWORDS = {
        'python': 1.5, 'machine learning': 1.5, 'deep learning': 1.5,
        'sql': 1.3, 'aws': 1.3, 'docker': 1.3, 'kubernetes': 1.3,
        'react': 1.2, 'java': 1.2, 'javascript': 1.2, 'tensorflow': 1.4,
        'pytorch': 1.4, 'data science': 1.3, 'nodejs': 1.2,
        'angular': 1.2, 'vue': 1.1, 'spring': 1.2, 'django': 1.3
    }

    # 6-Stage Recruitment Weights (Chen, 2022)
    STAGE_WEIGHTS = {
        'promotion':   0.10,
        'search':      0.10,
        'application': 0.15,
        'screening':   0.25,
        'assessment':  0.30,
        'coordination':0.10
    }

    # Model hyper-parameters
    KNN_NEIGHBORS    = 5
    W2V_VECTOR_SIZE  = 100
    W2V_WINDOW       = 5
    W2V_MIN_COUNT    = 2
    GRU_HIDDEN_SIZE  = 64
    LDA_N_TOPICS     = 10
    ATTENTION_HEADS  = 4

    # Evaluation
    TOP_K        = 10
    PRECISION_K  = 10
    CV_FOLDS     = 5
    TEST_SIZE    = 0.2
    RANDOM_STATE = 42

    # System
    USE_GPU         = torch.cuda.is_available()
    SEMANTIC_MODEL  = 'all-MiniLM-L6-v2'
    MAX_CV_LENGTH   = 15000
    BATCH_SIZE      = 32

    # SDG 8
    TARGET_AUDIENCE = "Young Bangladeshi students and fresh graduates"
    SDG_GOAL        = "SDG-8: Decent Work and Economic Growth"

print("[Done] Config loaded")
print(f"   GPU: {Config.USE_GPU} | Model: {Config.SEMANTIC_MODEL} | Top-K: {Config.TOP_K}")

✅ Config loaded
   GPU: True | Model: all-MiniLM-L6-v2 | Top-K: 10


## Step 3: Skill Database

Comprehensive skill catalogue with real course links from Coursera, Udemy, freeCodeCamp, and Kaggle.

In [5]:
class SkillDatabase:
    """Complete skill database with real course links."""

    SKILLS = {
        'python': {
            'category': 'Programming', 'difficulty': 'Medium', 'time': '3-6 months',
            'salary_impact': 15000, 'demand_score': 1.5,
            'courses': [
                {'name': 'Python for Everybody Specialization', 'platform': 'Coursera',
                 'url': 'https://www.coursera.org/specializations/python', 'price': 'Free (Audit)', 'duration': '8 months'},
                {'name': 'Complete Python Bootcamp', 'platform': 'Udemy',
                 'url': 'https://www.udemy.com/course/complete-python-bootcamp/', 'price': '$13.99', 'duration': '22 hours'},
                {'name': 'Python Tutorial', 'platform': 'freeCodeCamp',
                 'url': 'https://www.youtube.com/watch?v=rfscVS0vtbw', 'price': 'Free', 'duration': '4.5 hours'}
            ]
        },
        'java': {
            'category': 'Programming', 'difficulty': 'Medium', 'time': '3-6 months',
            'salary_impact': 14000, 'demand_score': 1.3,
            'courses': [
                {'name': 'Java Programming and Software Engineering', 'platform': 'Coursera',
                 'url': 'https://www.coursera.org/specializations/java-programming', 'price': 'Free (Audit)', 'duration': '6 months'}
            ]
        },
        'javascript': {
            'category': 'Web Development', 'difficulty': 'Medium', 'time': '2-4 months',
            'salary_impact': 13000, 'demand_score': 1.4,
            'courses': [
                {'name': 'JavaScript Algorithms and Data Structures', 'platform': 'freeCodeCamp',
                 'url': 'https://www.freecodecamp.org/learn/javascript-algorithms-and-data-structures/', 'price': 'Free', 'duration': '300 hours'}
            ]
        },
        'sql': {
            'category': 'Database', 'difficulty': 'Easy', 'time': '1-2 months',
            'salary_impact': 10000, 'demand_score': 1.3,
            'courses': [
                {'name': 'SQL for Data Science', 'platform': 'Coursera',
                 'url': 'https://www.coursera.org/learn/sql-for-data-science', 'price': 'Free (Audit)', 'duration': '4 weeks'}
            ]
        },
        'machine learning': {
            'category': 'AI/ML', 'difficulty': 'Hard', 'time': '6-12 months',
            'salary_impact': 25000, 'demand_score': 1.8,
            'courses': [
                {'name': 'Machine Learning Specialization', 'platform': 'Coursera',
                 'url': 'https://www.coursera.org/specializations/machine-learning-introduction', 'price': 'Free (Audit)', 'duration': '3 months'},
                {'name': 'Intro to Machine Learning', 'platform': 'Kaggle',
                 'url': 'https://www.kaggle.com/learn/intro-to-machine-learning', 'price': 'Free', 'duration': '3 hours'}
            ]
        },
        'deep learning': {
            'category': 'AI/ML', 'difficulty': 'Hard', 'time': '6-12 months',
            'salary_impact': 28000, 'demand_score': 1.7,
            'courses': [
                {'name': 'Deep Learning Specialization', 'platform': 'Coursera',
                 'url': 'https://www.coursera.org/specializations/deep-learning', 'price': 'Free (Audit)', 'duration': '5 months'}
            ]
        },
        'react': {
            'category': 'Web Development', 'difficulty': 'Medium', 'time': '2-4 months',
            'salary_impact': 15000, 'demand_score': 1.3,
            'courses': [
                {'name': 'Front-End Development with React', 'platform': 'freeCodeCamp',
                 'url': 'https://www.freecodecamp.org/learn/front-end-development-libraries/', 'price': 'Free', 'duration': '300 hours'}
            ]
        },
        'aws': {
            'category': 'Cloud Computing', 'difficulty': 'Medium', 'time': '3-6 months',
            'salary_impact': 20000, 'demand_score': 1.5,
            'courses': [
                {'name': 'AWS Fundamentals', 'platform': 'Coursera',
                 'url': 'https://www.coursera.org/specializations/aws-fundamentals', 'price': 'Free (Audit)', 'duration': '4 months'}
            ]
        },
        'docker': {
            'category': 'DevOps', 'difficulty': 'Medium', 'time': '1-3 months',
            'salary_impact': 12000, 'demand_score': 1.3,
            'courses': [
                {'name': 'Docker for Beginners', 'platform': 'YouTube',
                 'url': 'https://www.youtube.com/watch?v=fqMOX6JJhGo', 'price': 'Free', 'duration': '2 hours'}
            ]
        },
        'kubernetes': {
            'category': 'DevOps', 'difficulty': 'Hard', 'time': '3-6 months',
            'salary_impact': 22000, 'demand_score': 1.6,
            'courses': [
                {'name': 'Getting Started with Kubernetes', 'platform': 'Coursera',
                 'url': 'https://www.coursera.org/learn/google-kubernetes-engine', 'price': 'Free (Audit)', 'duration': '4 weeks'}
            ]
        },
        'tensorflow': {
            'category': 'AI/ML', 'difficulty': 'Hard', 'time': '4-8 months',
            'salary_impact': 24000, 'demand_score': 1.6,
            'courses': [
                {'name': 'TensorFlow Developer Certificate', 'platform': 'Coursera',
                 'url': 'https://www.coursera.org/professional-certificates/tensorflow-in-practice', 'price': 'Free (Audit)', 'duration': '4 months'}
            ]
        },
        'pytorch': {
            'category': 'AI/ML', 'difficulty': 'Hard', 'time': '4-8 months',
            'salary_impact': 24000, 'demand_score': 1.6,
            'courses': [
                {'name': 'Deep Learning with PyTorch', 'platform': 'Coursera',
                 'url': 'https://www.coursera.org/learn/deep-neural-networks-with-pytorch', 'price': 'Free (Audit)', 'duration': '5 weeks'}
            ]
        },
        'data science': {
            'category': 'Data Science', 'difficulty': 'Medium', 'time': '4-8 months',
            'salary_impact': 20000, 'demand_score': 1.5,
            'courses': [
                {'name': 'IBM Data Science Professional', 'platform': 'Coursera',
                 'url': 'https://www.coursera.org/professional-certificates/ibm-data-science', 'price': 'Free (Audit)', 'duration': '11 months'}
            ]
        }
    }

    @classmethod
    def get_skill_info(cls, skill_name: str) -> Dict:
        skill_lower = skill_name.lower()
        if skill_lower in cls.SKILLS:
            return cls.SKILLS[skill_lower]
        for key in cls.SKILLS:
            if key in skill_lower or skill_lower in key:
                return cls.SKILLS[key]
        return {
            'category': 'General', 'difficulty': 'Medium', 'time': '2-4 months',
            'salary_impact': 10000, 'demand_score': 1.0,
            'courses': [{
                'name': f'Learn {skill_name}', 'platform': 'YouTube',
                'url': f'https://www.youtube.com/results?search_query={skill_name.replace(" ", "+")}+tutorial',
                'price': 'Free', 'duration': 'Self-paced'
            }]
        }

    @classmethod
    def get_all_skills(cls) -> List[str]:
        return list(cls.SKILLS.keys())

print(f"[Done] SkillDatabase loaded — {len(SkillDatabase.SKILLS)} skills available")

✅ SkillDatabase loaded — 13 skills available


## Step 4: Data Models

Dataclasses for `CVProfile`, `JobPosting`, and `RecommendationResult`.

In [6]:
@dataclass
class CVProfile:
    """Complete CV/resume profile."""
    name: str
    email: str
    phone: str
    technical_skills: List[str]
    soft_skills: List[str]
    experience_years: float
    education_level: str
    certifications: List[str]
    projects: List[str]
    languages: List[str]
    raw_text: str
    preprocessed_text: str = ""
    sentiment_score: float = 0.0
    skill_embeddings: np.ndarray = field(default_factory=lambda: np.array([]))


@dataclass
class JobPosting:
    """Complete job posting."""
    job_id: str
    title: str
    company: str
    location: str
    description: str
    required_skills: List[str]
    preferred_skills: List[str]
    experience_required: float
    education_required: str
    salary_range: Tuple[float, float]
    employment_type: str
    preprocessed_description: str = ""
    skill_embeddings: np.ndarray = field(default_factory=lambda: np.array([]))


@dataclass
class RecommendationResult:
    """Job recommendation result."""
    job: JobPosting
    scores: Dict[str, float]
    ensemble_score: float
    rank: int
    confidence: str
    stage_scores: Dict[str, float]


print("[Done] Data models defined: CVProfile, JobPosting, RecommendationResult")

✅ Data models defined: CVProfile, JobPosting, RecommendationResult


## Step 5: Text Preprocessing & CV Parser

- `TextPreprocessor` — tokenization, stopword removal, lemmatization, sentiment analysis
- `CVParser` — extract structured profile from PDF, DOCX, or plain-text CV

In [7]:
class TextPreprocessor:
    """
    Complete NLP preprocessing pipeline.
    Research Proposal: Remove stopwords, tokenize, lemmatize, lowercase.
    """

    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        try:
            self.stop_words = set(stopwords.words('english'))
        except:
            self.stop_words = {
                'the', 'is', 'at', 'which', 'on', 'a', 'an', 'as', 'are',
                'was', 'were', 'been', 'be', 'have', 'has', 'had', 'do',
                'does', 'did', 'will', 'would', 'should', 'could', 'may'
            }
        try:
            self.sentiment_analyzer = SentimentIntensityAnalyzer()
        except:
            self.sentiment_analyzer = None

    def preprocess(self, text: str) -> str:
        if not text or not isinstance(text, str):
            return ""
        try:
            text = text.lower()
            text = re.sub(r'[^a-z0-9\s]', ' ', text)
            try:
                tokens = word_tokenize(text)
            except:
                tokens = text.split()
            tokens = [w for w in tokens if w not in self.stop_words and len(w) > 2]
            tokens = [self.lemmatizer.lemmatize(w) for w in tokens]
            return " ".join(tokens)
        except:
            return re.sub(r'[^a-z0-9\s]', ' ', text.lower())

    def extract_keywords(self, text: str, top_n: int = 20) -> List[str]:
        processed = self.preprocess(text)
        words = processed.split()
        word_freq = Counter(words)
        return [word for word, _ in word_freq.most_common(top_n)]

    def analyze_sentiment(self, text: str) -> float:
        if not self.sentiment_analyzer:
            return 0.0
        try:
            return self.sentiment_analyzer.polarity_scores(text)['compound']
        except:
            return 0.0


print("[Done] TextPreprocessor defined")

✅ TextPreprocessor defined


In [8]:
class CVParser:
    """
    Parse CV from PDF, DOCX, or TXT.
    Extracts: skills, experience, education, certifications, projects, languages.
    """

    TECH_SKILLS = [
        'python', 'java', 'javascript', 'typescript', 'c++', 'c#', 'ruby', 'go', 'rust', 'php',
        'sql', 'mysql', 'postgresql', 'mongodb', 'redis', 'oracle',
        'react', 'angular', 'vue', 'nodejs', 'express', 'django', 'flask', 'spring', 'laravel',
        'aws', 'azure', 'gcp', 'docker', 'kubernetes', 'jenkins', 'terraform',
        'machine learning', 'deep learning', 'data science', 'tensorflow', 'pytorch', 'scikit-learn',
        'git', 'linux', 'html', 'css', 'sass', 'rest api', 'graphql', 'microservices',
        'agile', 'scrum', 'jira', 'confluence'
    ]

    SOFT_SKILLS = [
        'leadership', 'communication', 'teamwork', 'problem solving',
        'creativity', 'adaptability', 'time management', 'critical thinking',
        'project management', 'analytical', 'collaborative', 'innovative'
    ]

    def __init__(self):
        self.preprocessor = TextPreprocessor()

    def extract_text_from_pdf(self, pdf_bytes: bytes) -> str:
        try:
            text = ""
            with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
                for page in pdf.pages[:15]:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
            return text[:Config.MAX_CV_LENGTH]
        except Exception as e:
            print(f"[Warning]  PDF extraction error: {e}")
            return ""

    def extract_text_from_docx(self, docx_bytes: bytes) -> str:
        if not DOCX_AVAILABLE:
            return ""
        try:
            doc = docx.Document(io.BytesIO(docx_bytes))
            return "\n".join(p.text for p in doc.paragraphs)[:Config.MAX_CV_LENGTH]
        except Exception as e:
            print(f"[Warning]  DOCX extraction error: {e}")
            return ""

    def extract_text_from_file(self, file_content: bytes, filename: str) -> str:
        fname = filename.lower()
        if fname.endswith('.pdf'):
            return self.extract_text_from_pdf(file_content)
        elif fname.endswith('.docx'):
            return self.extract_text_from_docx(file_content)
        else:
            try:
                return file_content.decode('utf-8', errors='ignore')[:Config.MAX_CV_LENGTH]
            except:
                return ""

    def parse_cv(self, cv_text: str) -> CVProfile:
        cv_lower = cv_text.lower()
        lines = [l.strip() for l in cv_text.split('\n') if l.strip()]

        name  = lines[0][:50] if lines else 'Candidate'
        email = (re.findall(r'\b[\w.%+-]+@[\w.-]+\.[A-Za-z]{2,}\b', cv_text) or [''])[0]
        phone = (re.findall(r'[\+\(]?[1-9][0-9 .\-\(\)]{8,}[0-9]', cv_text) or [''])[0]

        technical_skills = [s for s in self.TECH_SKILLS if re.search(r'\b' + re.escape(s) + r'\b', cv_lower)]
        soft_skills      = [s for s in self.SOFT_SKILLS if s in cv_lower]

        experience = 0.0
        for pat in [r'(\d+)\+?\s*(?:years?|yrs?)\s+(?:of\s+)?experience',
                    r'experience.*?(\d+)\+?\s*(?:years?|yrs?)',
                    r'(\d+)\+?\s*(?:years?|yrs?)\s+in']:
            m = re.findall(pat, cv_lower)
            if m:
                experience = max(float(x) for x in m)
                break

        if   any(w in cv_lower for w in ['phd', 'ph.d', 'doctorate']):
            education = 'PhD'
        elif any(w in cv_lower for w in ['master', 'mba', 'ms', 'm.s']):
            education = 'Masters'
        else:
            education = 'Bachelors'

        certifications = list(set(
            m.strip()[:100]
            for pat in [r'certified\s+([\w\s]+?)(?:\n|,|;)',
                        r'\b(aws|azure|pmp|cissp|cpa|cfa|comptia)[\s-]certified']
            for m in re.findall(pat, cv_text, re.IGNORECASE)
            if isinstance(m, str)
        ))[:10]

        projects = list(set(
            m.strip()[:200]
            for pat in [r'developed\s+(.*?)(?:\.|,|\n)',
                        r'built\s+(.*?)(?:\.|,|\n)',
                        r'created\s+(.*?)(?:\.|,|\n)']
            for m in re.findall(pat, cv_text, re.IGNORECASE)
            if isinstance(m, str) and len(m.strip()) > 10
        ))[:10]

        languages = ['English']
        lang_matches = re.findall(r'languages?:\s*(.*?)(?:\n\n|\n[A-Z])', cv_text, re.IGNORECASE)
        if lang_matches:
            for lang in ['bengali', 'hindi', 'spanish', 'french', 'german', 'chinese', 'arabic']:
                if lang in lang_matches[0].lower():
                    languages.append(lang.title())
        languages = list(set(languages))

        preprocessed = self.preprocessor.preprocess(cv_text)
        sentiment    = self.preprocessor.analyze_sentiment(cv_text)

        return CVProfile(
            name=name, email=email, phone=phone,
            technical_skills=technical_skills, soft_skills=soft_skills,
            experience_years=experience, education_level=education,
            certifications=certifications, projects=projects, languages=languages,
            raw_text=cv_text, preprocessed_text=preprocessed, sentiment_score=sentiment
        )


print("[Done] CVParser defined")

✅ CVParser defined


## Step 6: ML Models

All seven models:
1. **Baseline Keyword Model** — simple overlap (comparison baseline)
2. **TF-IDF + Cosine Similarity** — domain-weighted (Ajjam & Al-Raweshidy, 2026)
3. **Word2Vec** — semantic word embeddings (Alsaif et al., 2022)
4. **Jaccard Coefficient** — skill-set overlap (Alsaif et al., 2022)
5. **K-Nearest Neighbors** — profile-based similarity
6. **Linear Regression** — salary prediction
7. **6-Stage Recruitment Engine** — process-aware scoring (Chen, 2022)

In [9]:
# Model 1: Baseline Keyword Model

class BaselineKeywordModel:
    """Simple keyword overlap — used as performance comparison baseline."""

    def match(self, cv_text: str, job_description: str) -> float:
        cv_words  = set(cv_text.lower().split())
        job_words = set(job_description.lower().split())
        if not job_words:
            return 0.0
        return len(cv_words & job_words) / len(job_words)

    def recommend(self, cv_profile: CVProfile, jobs: List[JobPosting], top_k: int = 10) -> List[Dict]:
        results = [{
            'job_id': job.job_id, 'title': job.title,
            'score': self.match(cv_profile.raw_text, job.description),
            'method': 'baseline_keyword'
        } for job in jobs]
        results.sort(key=lambda x: x['score'], reverse=True)
        return results[:top_k]


print("[Done] BaselineKeywordModel defined")

✅ BaselineKeywordModel defined


In [10]:
# Model 2: TF-IDF + Cosine Similarity

class TFIDFMatcher:
    """
    TF-IDF with domain-specific keyword weighting.
    Reference: Ajjam & Al-Raweshidy, 2026
    """

    def __init__(self):
        self.vectorizer  = None
        self.preprocessor = TextPreprocessor()

    def fit_transform(self, documents: List[str]) -> np.ndarray:
        if not documents:
            return np.array([])
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, min_df=1, norm='l2')
        tfidf_matrix = self.vectorizer.fit_transform(documents)
        # Domain keyword boosting
        try:
            feature_names = self.vectorizer.get_feature_names_out()
            for i, term in enumerate(feature_names):
                if term.lower() in Config.DOMAIN_KEYWORDS:
                    tfidf_matrix[:, i] = tfidf_matrix[:, i] * Config.DOMAIN_KEYWORDS[term.lower()]
        except:
            pass
        return tfidf_matrix.toarray()

    def calculate_similarity(self, cv_vector: np.ndarray, job_vectors: np.ndarray) -> np.ndarray:
        if cv_vector.size == 0 or job_vectors.size == 0:
            return np.array([])
        return cosine_similarity(cv_vector.reshape(1, -1), job_vectors)[0]

    def calculate_precision_recall_at_k(self, similarities, relevant_indices, k=10):
        if len(similarities) == 0:
            return 0.0, 0.0
        top_k = set(similarities.argsort()[-k:][::-1])
        rel   = set(relevant_indices)
        inter = len(rel & top_k)
        return inter / k, inter / len(rel) if rel else 0.0


print("[Done] TFIDFMatcher defined")

✅ TFIDFMatcher defined


In [11]:
# Model 3: Word2Vec

class Word2VecMatcher:
    """
    Word2Vec semantic embeddings.
    Reference: Alsaif et al., 2022
    """

    def __init__(self):
        self.model      = None
        self.is_trained = False

    def train(self, documents: List[str]):
        print(" Training Word2Vec...")
        sentences = [doc.lower().split() for doc in documents if doc]
        if not sentences:
            print("[Warning]  No sentences to train Word2Vec")
            return
        try:
            self.model = Word2Vec(
                sentences=sentences,
                vector_size=Config.W2V_VECTOR_SIZE,
                window=Config.W2V_WINDOW,
                min_count=Config.W2V_MIN_COUNT,
                workers=4, epochs=5, sg=0
            )
            self.is_trained = True
            print(f"   [Done] Vocabulary: {len(self.model.wv)} words")
        except Exception as e:
            print(f"[Warning]  Word2Vec training failed: {e}")

    def get_document_embedding(self, text: str) -> np.ndarray:
        if not self.is_trained:
            return np.zeros(Config.W2V_VECTOR_SIZE)
        vecs = [self.model.wv[w] for w in text.lower().split() if w in self.model.wv]
        return np.mean(vecs, axis=0) if vecs else np.zeros(Config.W2V_VECTOR_SIZE)

    def calculate_similarity(self, text1: str, text2: str) -> float:
        e1, e2 = self.get_document_embedding(text1), self.get_document_embedding(text2)
        n1, n2 = np.linalg.norm(e1), np.linalg.norm(e2)
        if n1 == 0 or n2 == 0:
            return 0.0
        return float(np.clip(np.dot(e1, e2) / (n1 * n2), 0.0, 1.0))


print("[Done] Word2VecMatcher defined")

✅ Word2VecMatcher defined


In [21]:
# Model 4: Jaccard Coefficient

class JaccardMatcher:
    """Jaccard coefficient for skill-set similarity. Reference: Alsaif et al., 2022"""

    @staticmethod
    def jaccard_similarity(set1: set, set2: set) -> float:
        if not set1 or not set2:
            return 0.0
        inter = len(set1 & set2)
        union = len(set1 | set2)
        return inter / union if union else 0.0

    def calculate_skill_match(self, cv_skills, job_skills):
        return self.jaccard_similarity(
            set(s.lower().strip() for s in cv_skills if s),
            set(s.lower().strip() for s in job_skills if s)
        )

    def calculate_text_similarity(self, text1: str, text2: str) -> float:
        return self.jaccard_similarity(set(text1.lower().split()), set(text2.lower().split()))


# Model 5: K-Nearest Neighbors

class KNNRecommender:
    """KNN for finding similar job profiles."""

    def __init__(self, n_neighbors: int = Config.KNN_NEIGHBORS):
        self.n_neighbors = n_neighbors
        self.model       = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine', algorithm='brute')
        self.jobs        = None
        self.is_fitted   = False

    def fit(self, job_vectors: np.ndarray, jobs: List[JobPosting]):
        if len(jobs) < self.n_neighbors:
            print(f"[Warning]  Need ≥{self.n_neighbors} jobs for KNN (got {len(jobs)})")
            return
        try:
            self.jobs = jobs
            self.model.fit(job_vectors)
            self.is_fitted = True
        except Exception as e:
            print(f"[Warning]  KNN fitting failed: {e}")

    def recommend(self, cv_vector: np.ndarray, top_k: int = 10) -> List[Dict]:
        if not self.is_fitted:
            return []
        try:
            k = min(top_k, len(self.jobs))
            dists, idxs = self.model.kneighbors(cv_vector.reshape(1, -1), n_neighbors=k)
            return [{
                'job_id': self.jobs[idx].job_id,
                'title':  self.jobs[idx].title,
                'score':  1.0 / (1.0 + dist),
                'distance': float(dist),
                'method': 'knn'
            } for dist, idx in zip(dists[0], idxs[0])]
        except Exception as e:
            print(f"[Warning]  KNN recommendation failed: {e}")
            return []


# Model 6: Salary Predictor (Linear Regression)

class SalaryPredictor:
    """Linear regression salary prediction using skills, experience, and education."""

    def __init__(self):
        self.model      = LinearRegression()
        self.scaler     = StandardScaler()
        self.is_trained = False

    def _generate_training_data(self, n=1000):
        np.random.seed(Config.RANDOM_STATE)
        skills  = np.random.randint(1, 15, n)
        exp     = np.random.randint(0, 20, n)
        edu     = np.random.choice([1, 2, 3], n, p=[0.6, 0.3, 0.1])
        # FIXED: Explicitly cast salary to float64 to allow addition of float noise
        salary  = 45000.0 + (skills * 2500.0) + (exp * 3500.0) + (edu * 7000.0)
        salary += np.random.normal(0, 5000, n)
        salary  = np.clip(salary, 35000, 200000)
        return np.column_stack([skills, exp, edu]), salary

    def train(self):
        print(" Training Salary Predictor...")
        X, y = self._generate_training_data()
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=Config.TEST_SIZE, random_state=Config.RANDOM_STATE)
        self.scaler.fit(Xtr)
        self.model.fit(self.scaler.transform(Xtr), ytr)
        pred = self.model.predict(self.scaler.transform(Xte))
        self.is_trained = True
        r2  = r2_score(yte, pred)
        mae = mean_absolute_error(yte, pred)
        print(f"   [Done] R²={r2:.3f}, MAE=${mae:,.0f}")
        return {'r2': r2, 'mae': mae}

    def predict(self, cv: CVProfile) -> Tuple[float, Dict]:
        if not self.is_trained:
            self.train()
        edu_map = {'Bachelors': 1, 'Masters': 2, 'PhD': 3}
        n_skills = len(set(cv.technical_skills + cv.soft_skills))
        feats    = np.array([[n_skills, cv.experience_years, edu_map.get(cv.education_level, 1)]])
        pred     = self.model.predict(self.scaler.transform(feats))[0]
        margin   = pred * 0.15
        return pred, {'predicted': pred, 'lower_bound': pred - margin,
                      'upper_bound': pred + margin,
                      'range': f"${pred - margin:,.0f} – ${pred + margin:,.0f}"}

In [13]:
# Model 7: 6-Stage Recruitment Engine

class SixStageRecruitmentEngine:
    """
    6-stage recruitment process scoring.
    Reference: Chen (2022) — promotion, search, application, screening, assessment, coordination
    """

    def calculate_all_stage_scores(self, cv: CVProfile, job: JobPosting) -> Dict[str, float]:
        return {
            'promotion':    self._stage1(cv, job),
            'search':       self._stage2(cv, job),
            'application':  self._stage3(cv, job),
            'screening':    self._stage4(cv, job),
            'assessment':   self._stage5(cv, job),
            'coordination': self._stage6(cv, job),
        }

    def _stage1(self, cv, job):
        cv_kw  = set(cv.preprocessed_text.split()[:50])
        job_kw = set(job.preprocessed_description.split()[:50])
        return min(0.5 + len(cv_kw & job_kw) / 20, 1.0)

    def _stage2(self, cv, job):
        title_words = job.title.lower().split()
        hits = sum(1 for w in title_words if w in cv.raw_text.lower())
        return min(hits / max(len(title_words), 1) * 0.5 + (0.5 if cv.technical_skills else 0), 1.0)

    def _stage3(self, cv, job):
        s  = 0.2 if cv.email else 0.0
        s += 0.1 if cv.phone else 0.0
        s += 0.3 if len(cv.technical_skills) >= 3 else 0.0
        s += 0.2 if cv.projects else 0.0
        s += 0.2 if cv.certifications else 0.0
        return min(s, 1.0)

    def _stage4(self, cv, job):
        edu_map = {'High School': 0, 'Bachelors': 1, 'Masters': 2, 'PhD': 3}
        s  = 0.4 if cv.experience_years >= job.experience_required * 0.7 else 0.0
        s += 0.4 if edu_map.get(cv.education_level, 0) >= edu_map.get(job.education_required, 1) else 0.0
        s += 0.2 if cv.technical_skills else 0.0
        return min(s, 1.0)

    def _stage5(self, cv, job):
        edu_map = {'High School': 0, 'Bachelors': 1, 'Masters': 2, 'PhD': 3}
        exp_r = min(cv.experience_years / max(job.experience_required, 0.5), 1.5)
        edu_r = min(edu_map.get(cv.education_level, 0) / max(edu_map.get(job.education_required, 1), 1), 1.5)
        skill_s = 0.3 if len(cv.technical_skills) >= 5 else (0.2 if len(cv.technical_skills) >= 3 else 0.0)
        return min((exp_r / 1.5) * 0.4 + (edu_r / 1.5) * 0.3 + skill_s, 1.0)

    def _stage6(self, cv, job):
        s  = 0.7
        s += 0.15 if (cv.email and cv.phone) else 0.0
        s += 0.15 if len(cv.languages) > 1 else 0.0
        return min(s, 1.0)

    def calculate_overall_score(self, stage_scores: Dict[str, float]) -> float:
        return sum(score * Config.STAGE_WEIGHTS.get(stage, 0.0)
                   for stage, score in stage_scores.items())


print("[Done] SixStageRecruitmentEngine defined")

✅ SixStageRecruitmentEngine defined


## Step 7: Evaluation Metrics

Precision@K, Recall@K, F1, Wilcoxon Test, Cohen's D.

In [14]:
class EvaluationMetrics:
    """Complete evaluation suite from the research proposal."""

    @staticmethod
    def precision_at_k(y_true, y_pred, k=10):
        if not y_pred or k == 0:
            return 0.0
        return sum(1 for item in y_pred[:k] if item in y_true) / k

    @staticmethod
    def recall_at_k(y_true, y_pred, k=10):
        if not y_true:
            return 0.0
        return sum(1 for item in y_pred[:k] if item in y_true) / len(y_true)

    @staticmethod
    def f1_score_at_k(precision, recall):
        if precision + recall == 0:
            return 0.0
        return 2 * precision * recall / (precision + recall)

    @staticmethod
    def wilcoxon_test(scores1, scores2):
        try:
            if len(scores1) != len(scores2) or len(scores1) < 5:
                return 0.0, 1.0
            stat, p = wilcoxon(scores1, scores2)
            return float(stat), float(p)
        except:
            return 0.0, 1.0

    @staticmethod
    def cohens_d(scores1, scores2):
        try:
            if not scores1 or not scores2:
                return 0.0
            n1, n2 = len(scores1), len(scores2)
            v1 = np.var(scores1, ddof=1) if n1 > 1 else 0
            v2 = np.var(scores2, ddof=1) if n2 > 1 else 0
            pool = np.sqrt(((n1-1)*v1 + (n2-1)*v2) / (n1+n2-2))
            return (np.mean(scores1) - np.mean(scores2)) / pool if pool else 0.0
        except:
            return 0.0


print("[Done] EvaluationMetrics defined")

✅ EvaluationMetrics defined


## Step 8: CSV Data Loader

Loads the three CSV datasets. Falls back to 15 built-in demo jobs if no files are uploaded.

In [15]:
class CSVDataLoader:
    """
    Loads the 3 CSV datasets:
    1. JobsFE.csv — 8 columns, ~10 000 entries
    2. career_guidance_dataset.csv — 22 columns, ~1 000 entries
    3. job_recommendation_dataset.csv — 7 columns, ~50 000 entries
    Falls back to demo data if files are not present.
    """

    def __init__(self):
        self.jobs         = []
        self.preprocessor = TextPreprocessor()

    def load_csv_files(self, file_paths: Dict[str, str] = None):
        print("\n LOADING CSV FILES...")
        print("=" * 60)
        if file_paths:
            for name, path in file_paths.items():
                try:
                    df = pd.read_csv(path)
                    print(f"[Done] {name}: {len(df):,} rows × {len(df.columns)} cols")
                    self._extract_jobs(df, name)
                except Exception as e:
                    print(f"[Warning]  {name}: {e}")

        if not self.jobs:
            print("\n[Warning]  No CSV files loaded — using demo data.")
            self._create_demo_jobs()

        print(f"\n[Done] Total jobs available: {len(self.jobs):,}")

    def _extract_jobs(self, df: pd.DataFrame, source_name: str):
        title_col   = next((c for c in df.columns if any(k in c.lower() for k in ['title','job','position','role'])), None)
        desc_col    = next((c for c in df.columns if any(k in c.lower() for k in ['desc','requirement','detail','summary'])), title_col)
        company_col = next((c for c in df.columns if any(k in c.lower() for k in ['company','employer','organization'])), None)
        loc_col     = next((c for c in df.columns if any(k in c.lower() for k in ['location','city','place'])), None)
        if not title_col:
            return
        for _, row in df.iterrows():
            try:
                title = str(row[title_col])
                if not title or title == 'nan' or len(title) < 3:
                    continue
                desc    = str(row[desc_col])    if desc_col    else title
                company = str(row[company_col]) if company_col else 'Company'
                loc     = str(row[loc_col])     if loc_col     else 'Remote'
                self.jobs.append(JobPosting(
                    job_id=f"{source_name}_{len(self.jobs)+1}",
                    title=title[:200], company=(company if company != 'nan' else 'Company')[:100],
                    location=(loc if loc != 'nan' else 'Remote')[:100],
                    description=desc[:2000], required_skills=[], preferred_skills=[],
                    experience_required=2.0, education_required='Bachelors',
                    salary_range=(50000, 95000), employment_type='Full-time',
                    preprocessed_description=self.preprocessor.preprocess(desc)
                ))
            except:
                continue

    def _create_demo_jobs(self):
        DEMO = [
            ("Software Engineer",          "Python machine learning SQL AWS Docker Kubernetes REST API microservices"),
            ("Data Scientist",             "data analysis statistics Python machine learning TensorFlow pandas numpy visualization"),
            ("Full Stack Developer",        "React Node.js JavaScript MongoDB REST API frontend backend TypeScript"),
            ("Machine Learning Engineer",   "machine learning deep learning TensorFlow PyTorch scikit-learn NLP computer vision"),
            ("DevOps Engineer",             "Docker Kubernetes CI/CD Jenkins AWS Azure automation monitoring deployment"),
            ("Backend Developer",           "Python Java SQL database microservices REST GraphQL Django Flask"),
            ("Frontend Developer",          "React JavaScript HTML CSS Angular Vue TypeScript UI/UX responsive"),
            ("Data Analyst",               "data analysis SQL Python Excel Power BI Tableau dashboards reporting"),
            ("Cloud Engineer",             "AWS cloud Lambda EC2 S3 infrastructure scaling security monitoring"),
            ("Business Analyst",           "requirements gathering data visualization SQL Excel stakeholder management"),
            ("Mobile Developer",           "React Native Flutter iOS Android JavaScript Swift Kotlin"),
            ("Security Engineer",          "cybersecurity penetration testing vulnerability CISSP network security"),
            ("Product Manager",            "product management agile scrum roadmap user research prioritization"),
            ("QA Engineer",                "quality assurance Selenium pytest unit testing integration CI/CD"),
            ("System Administrator",       "Linux Windows server networking bash scripting monitoring troubleshooting"),
        ]
        for i, (title, desc) in enumerate(DEMO):
            self.jobs.append(JobPosting(
                job_id=f"demo_{i+1}", title=title, company="Tech Company BD",
                location="Remote / Bangladesh", description=desc,
                required_skills=[], preferred_skills=[],
                experience_required=2.0 + (i % 5),
                education_required="Masters" if i % 3 == 0 else "Bachelors",
                salary_range=(45000 + i*3000, 90000 + i*5000), employment_type="Full-time",
                preprocessed_description=self.preprocessor.preprocess(desc)
            ))


print("[Done] CSVDataLoader defined")

✅ CSVDataLoader defined


## Step 9: SkillBridge AI — Main Recommendation Engine

Ensemble of all 7 models with weighted score combination.

In [16]:
class SkillBridgeAI:
    """SkillBridge AI — ensemble of all ML models."""

    def __init__(self):
        print("=" * 70)
        print(" SKILLBRIDGE AI — ULTIMATE ML SYSTEM v13.0")
        print(f" {Config.TARGET_AUDIENCE}")
        print(f" {Config.SDG_GOAL}")
        print("=" * 70)

        self.preprocessor       = TextPreprocessor()
        self.cv_parser          = CVParser()
        self.data_loader        = CSVDataLoader()
        self.baseline_model     = BaselineKeywordModel()
        self.tfidf_matcher      = TFIDFMatcher()
        self.word2vec_matcher   = Word2VecMatcher()
        self.jaccard_matcher    = JaccardMatcher()
        self.knn_recommender    = KNNRecommender()
        self.salary_predictor   = SalaryPredictor()
        self.recruitment_engine = SixStageRecruitmentEngine()
        self.evaluator          = EvaluationMetrics()
        self.jobs               = []

    # Data
    def load_data(self, csv_files=None):
        self.data_loader.load_csv_files(csv_files)
        self.jobs = self.data_loader.jobs

    # Training
    def train_models(self):
        print("\n TRAINING ALL ML MODELS...")
        if not self.jobs:
            print("[Warning]  No jobs available"); return
        self.word2vec_matcher.train([j.preprocessed_description for j in self.jobs])
        self.salary_predictor.train()
        print("[Done] All models trained!")

    # CV Processing
    def process_cv(self, cv_content: bytes, filename: str) -> CVProfile:
        print("\n PROCESSING CV...")
        text = self.cv_parser.extract_text_from_file(cv_content, filename)
        if not text:
            print("[Warning]  Could not extract text from CV"); return None
        profile = self.cv_parser.parse_cv(text)
        print(f"[Done] {profile.name} | Skills: {len(profile.technical_skills+profile.soft_skills)} | Exp: {profile.experience_years}y | Edu: {profile.education_level}")
        return profile

    # Ensemble Recommendation
    def recommend_jobs(self, cv_profile: CVProfile, top_k: int = Config.TOP_K) -> Dict:
        print("\n GENERATING RECOMMENDATIONS...")
        if not self.jobs:
            print("[Warning]  No jobs available"); return {'recommendations': [], 'metrics': {}}

        # 1. Baseline
        baseline_recs   = self.baseline_model.recommend(cv_profile, self.jobs, len(self.jobs))
        baseline_scores = [r['score'] for r in baseline_recs]

        # 2. TF-IDF
        all_texts    = [cv_profile.preprocessed_text] + [j.preprocessed_description for j in self.jobs]
        tfidf_matrix = self.tfidf_matcher.fit_transform(all_texts)
        if tfidf_matrix.size > 0:
            cv_vec    = tfidf_matrix[0:1]
            job_vecs  = tfidf_matrix[1:]
            tfidf_s   = self.tfidf_matcher.calculate_similarity(cv_vec, job_vecs)
        else:
            cv_vec, job_vecs, tfidf_s = None, None, np.zeros(len(self.jobs))

        # 3. Word2Vec
        w2v_s = [self.word2vec_matcher.calculate_similarity(
                    cv_profile.preprocessed_text, j.preprocessed_description)
                 for j in self.jobs]

        # 4. Jaccard
        jac_s = [self.jaccard_matcher.calculate_text_similarity(
                    cv_profile.preprocessed_text, j.preprocessed_description)
                 for j in self.jobs]

        # 5. KNN
        if cv_vec is not None and len(self.jobs) >= Config.KNN_NEIGHBORS:
            self.knn_recommender.fit(job_vecs, self.jobs)
            knn_recs  = self.knn_recommender.recommend(cv_vec, len(self.jobs))
            knn_map   = {r['job_id']: r['score'] for r in knn_recs}
            knn_s     = [knn_map.get(j.job_id, 0.0) for j in self.jobs]
        else:
            knn_s = list(tfidf_s)

        # 6. 6-Stage Recruitment
        stage_s = []
        for job in self.jobs:
            ss = self.recruitment_engine.calculate_all_stage_scores(cv_profile, job)
            stage_s.append(self.recruitment_engine.calculate_overall_score(ss))

        # Ensemble
        final = []
        for i, job in enumerate(self.jobs):
            ens = (
                0.08 * (baseline_scores[i] if i < len(baseline_scores) else 0.0) +
                0.30 * float(tfidf_s[i] if i < len(tfidf_s) else 0.0) +
                0.20 * float(w2v_s[i]) +
                0.12 * float(jac_s[i]) +
                0.20 * float(knn_s[i] if i < len(knn_s) else 0.0) +
                0.10 * float(stage_s[i])
            )
            conf = "High" if ens > 0.75 else ("Medium" if ens > 0.55 else "Low")
            final.append({
                'job_id': job.job_id, 'title': job.title,
                'company': job.company, 'location': job.location,
                'ensemble_score': ens, 'confidence': conf,
                'salary_range': f"${job.salary_range[0]:,} – ${job.salary_range[1]:,}",
                'experience_required': job.experience_required,
                'description': job.description[:150] + '...',
                'scores': {
                    'baseline':    baseline_scores[i] if i < len(baseline_scores) else 0.0,
                    'tfidf':       float(tfidf_s[i] if i < len(tfidf_s) else 0.0),
                    'word2vec':    float(w2v_s[i]),
                    'jaccard':     float(jac_s[i]),
                    'knn':         float(knn_s[i] if i < len(knn_s) else 0.0),
                    'recruitment': float(stage_s[i])
                }
            })

        final.sort(key=lambda x: x['ensemble_score'], reverse=True)

        # Metrics
        pred_salary, salary_info = self.salary_predictor.predict(cv_profile)
        wstat, wp = self.evaluator.wilcoxon_test(list(tfidf_s[:20]), baseline_scores[:20])
        cd        = self.evaluator.cohens_d(list(tfidf_s[:20]), baseline_scores[:20])
        rel_idx   = [i for i, r in enumerate(final) if r['ensemble_score'] > 0.6]
        top_idx   = list(range(min(Config.PRECISION_K, len(final))))
        prec  = self.evaluator.precision_at_k(rel_idx, top_idx, Config.PRECISION_K)
        rec   = self.evaluator.recall_at_k(rel_idx, top_idx, Config.PRECISION_K)
        f1    = self.evaluator.f1_score_at_k(prec, rec)

        print(f"\n Salary: ${pred_salary:,.0f} | P@{Config.PRECISION_K}: {prec:.3f} | R@{Config.PRECISION_K}: {rec:.3f} | F1: {f1:.3f}")
        print(f"   Wilcoxon p={wp:.4f} | Cohen's D={cd:.3f}")
        print("[Done] Recommendations ready!")

        return {
            'cv_profile': cv_profile,
            'recommendations': final[:top_k],
            'metrics': {
                'predicted_salary': pred_salary, 'salary_info': salary_info,
                'precision_at_k': prec, 'recall_at_k': rec, 'f1_score': f1,
                'wilcoxon_p': wp, 'cohens_d': cd,
                'num_jobs': len(self.jobs), 'num_models': 6
            }
        }

    # Skill Recommendations
    def recommend_skills(self, cv_profile: CVProfile, top_k: int = 15) -> List[Dict]:
        print("\n GENERATING SKILL RECOMMENDATIONS...")
        user_skills = set(s.lower() for s in cv_profile.technical_skills + cv_profile.soft_skills)
        recs = []
        for skill in SkillDatabase.get_all_skills():
            if skill in user_skills:
                continue
            info = SkillDatabase.get_skill_info(skill)
            priority = int(min(info['demand_score'] * 30 + info['salary_impact'] / 1000 * 2, 100))
            recs.append({
                'skill': skill.title(), 'category': info['category'],
                'difficulty': info['difficulty'], 'time': info['time'],
                'salary_impact': info['salary_impact'], 'priority': priority,
                'courses': info['courses']
            })
        recs.sort(key=lambda x: x['priority'], reverse=True)
        print(f"[Done] {len(recs[:top_k])} skills recommended")
        return recs[:top_k]


print("[Done] SkillBridgeAI engine defined")

✅ SkillBridgeAI engine defined


## Step 10: Report Generator

Prints a complete formatted report: candidate profile, ML predictions, evaluation metrics, and top job recommendations.

In [17]:
class ReportGenerator:
    """Generate a comprehensive analysis report."""

    @staticmethod
    def generate(results: Dict):
        cv      = results['cv_profile']
        recs    = results['recommendations']
        metrics = results['metrics']
        LINE    = "=" * 70

        print(f"\n{LINE}")
        print(" SKILLBRIDGE AI — JOB & SKILL RECOMMENDATION REPORT")
        print(f"AI-Based Career Guidance System | {Config.SDG_GOAL}")
        print(LINE)

        # Candidate Profile
        print(f"\n{'-'*70}")
        print(" CANDIDATE PROFILE")
        print(f"{'-'*70}")
        print(f"Name:           {cv.name}")
        if cv.email: print(f"Email:          {cv.email}")
        if cv.phone: print(f"Phone:          {cv.phone}")
        print(f"Experience:     {cv.experience_years} years")
        print(f"Education:      {cv.education_level}")
        print(f"Tech Skills:    {', '.join(cv.technical_skills[:10]) or 'None detected'}")
        if cv.soft_skills:    print(f"Soft Skills:    {', '.join(cv.soft_skills[:5])}")
        if cv.certifications: print(f"Certifications: {', '.join(cv.certifications[:3])}")
        if cv.languages:      print(f"Languages:      {', '.join(cv.languages)}")

        # Predictions
        print(f"\n{'-'*70}")
        print(" ML PREDICTIONS")
        print(f"{'-'*70}")
        print(f"Predicted Salary:  ${metrics['predicted_salary']:,.0f}")
        print(f"Salary Range:      {metrics['salary_info']['range']}")
        print(f"Models Used:       {metrics['num_models']}")
        print(f"Jobs Analysed:     {metrics['num_jobs']:,}")

        # Evaluation
        print(f"\n{'-'*70}")
        print(" EVALUATION METRICS")
        print(f"{'-'*70}")
        print(f"Precision@{Config.PRECISION_K}:   {metrics['precision_at_k']:.3f}")
        print(f"Recall@{Config.PRECISION_K}:      {metrics['recall_at_k']:.3f}")
        print(f"F1-Score:          {metrics['f1_score']:.3f}")
        sig = " [OK] Statistically significant!" if metrics['wilcoxon_p'] < 0.05 else ""
        print(f"Wilcoxon p-value:  {metrics['wilcoxon_p']:.4f}{sig}")
        eff = " [OK] Large effect size!" if abs(metrics['cohens_d']) > 0.8 else ""
        print(f"Cohen's D:         {metrics['cohens_d']:.3f}{eff}")

        # Top Jobs
        print(f"\n{'-'*70}")
        print(f" TOP {min(10, len(recs))} JOB RECOMMENDATIONS")
        print(f"{'-'*70}")
        for i, job in enumerate(recs[:10], 1):
            print(f"\n{i}. {job['title']}")
            print(f"   {job['company']} | {job['location']}")
            print(f"   Ensemble: {job['ensemble_score']:.3f} ({job['confidence']} confidence)")
            s = job['scores']
            print(f"   TF-IDF:{s['tfidf']:.3f}  W2V:{s['word2vec']:.3f}  KNN:{s['knn']:.3f}  "
                  f"Jaccard:{s['jaccard']:.3f}  Recruit:{s['recruitment']:.3f}")
            print(f"    {job['salary_range']}  |   Exp: {job['experience_required']}y")

        # SDG 8
        print(f"\n{'-'*70}")
        print(" SDG-8 IMPACT")
        print(f"{'-'*70}")
        print(f"Target: {Config.TARGET_AUDIENCE}")
        for item in [
            "AI-powered job matching reduces youth unemployment",
            "Skill gap analysis guides career development",
            "Bias-free recommendations promote equal opportunity",
            "Salary transparency enables informed career decisions",
            "Free course links remove financial barriers to learning"
        ]:
            print(f"  [OK] {item}")

        print(f"\n{LINE}")
        print(f" Report: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"{LINE}\n")


print("[Done] ReportGenerator defined")

✅ ReportGenerator defined


## ▶️ Step 11: Run the Full Pipeline

Upload your CSV files and CV when prompted, or skip to use built-in demo data.

In [23]:
# STEP 11A: Initialise & Load Data

system = SkillBridgeAI()

# We already have the filenames from the previous run
csv_files = {
    'career_guidance_dataset.csv': 'career_guidance_dataset.csv',
    'job_recommendation_dataset.csv': 'job_recommendation_dataset.csv',
    'JobsFE.csv': 'JobsFE.csv'
}

system.load_data(csv_files)

🚀 SKILLBRIDGE AI — ULTIMATE ML SYSTEM v13.0
🎯 Young Bangladeshi students and fresh graduates
🌍 SDG-8: Decent Work and Economic Growth

📁 LOADING CSV FILES...
✅ career_guidance_dataset.csv: 1,000 rows × 22 cols
✅ job_recommendation_dataset.csv: 50,000 rows × 7 cols
✅ JobsFE.csv: 10,000 rows × 8 cols

✅ Total jobs available: 60,010


In [24]:
# STEP 11B: Train ML Models

system.train_models()


🤖 TRAINING ALL ML MODELS...
🔤 Training Word2Vec...
   ✅ Vocabulary: 543 words
💰 Training Salary Predictor...
   ✅ R²=0.962, MAE=$3,664
✅ All models trained!


In [26]:
# STEP 11C: Upload or Use Sample CV

# Using sample CV for immediate execution
SAMPLE_CV = """Md. Tanvir Rahman
Software Engineer
Email: tanvir.rahman@email.com
Phone: +880 1712-345678

PROFESSIONAL SUMMARY
Motivated software engineer with 3 years of experience in Python development,
machine learning, and web applications. Passionate about AI and data science.

WORK EXPERIENCE
Software Engineer | Tech Solutions BD, Dhaka (2021-2024)
• Developed Python applications for data analysis and visualization
• Built machine learning models using scikit-learn and TensorFlow
• Worked with SQL databases (PostgreSQL, MySQL)
• Deployed applications on AWS cloud platform

EDUCATION
Bachelor of Science in Computer Science — University of Dhaka (2020) CGPA 3.75/4.00

TECHNICAL SKILLS
Python, Java, JavaScript, SQL, React, Node.js, HTML, CSS, TensorFlow, AWS, Docker
"""
cv_profile = system.cv_parser.parse_cv(SAMPLE_CV)
print(f"[Done] CV loaded for: {cv_profile.name}")

✅ CV loaded for: Md. Tanvir Rahman


In [27]:
# STEP 11D: Generate Job Recommendations

results = system.recommend_jobs(cv_profile, top_k=15)


🎯 GENERATING RECOMMENDATIONS...

📊 Salary: $99,613 | P@10: 0.000 | R@10: 0.000 | F1: 0.000
   Wilcoxon p=0.0000 | Cohen's D=-190.860
✅ Recommendations ready!


In [28]:
# STEP 11E: Skill Development Recommendations

skill_recs = system.recommend_skills(cv_profile, top_k=10)


🎓 GENERATING SKILL RECOMMENDATIONS...
✅ 3 skills recommended


In [29]:
# STEP 11F: Full Report

ReportGenerator.generate(results)


📊 SKILLBRIDGE AI — JOB & SKILL RECOMMENDATION REPORT
AI-Based Career Guidance System | SDG-8: Decent Work and Economic Growth

──────────────────────────────────────────────────────────────────────
👤 CANDIDATE PROFILE
──────────────────────────────────────────────────────────────────────
Name:           Md. Tanvir Rahman
Email:          tanvir.rahman@email.com
Phone:          +880 1712-345678
Experience:     3.0 years
Education:      Bachelors
Tech Skills:    python, java, javascript, sql, mysql, postgresql, react, aws, docker, machine learning
Languages:      English

──────────────────────────────────────────────────────────────────────
🤖 ML PREDICTIONS
──────────────────────────────────────────────────────────────────────
Predicted Salary:  $99,613
Salary Range:      $84,671 – $114,555
Models Used:       6
Jobs Analysed:     60,010

──────────────────────────────────────────────────────────────────────
📈 EVALUATION METRICS
───────────────────────────────────────────────────────────

## Step 12: Visualizations

Interactive charts for score comparison, skill gap, and salary prediction.

In [31]:
# Chart 1: Ensemble Score for Top 10 Jobs

recs = results['recommendations'][:10]

fig = go.Figure(go.Bar(
    x=[r['ensemble_score'] for r in recs],
    y=[r['title'] for r in recs],
    orientation='h',
    marker=dict(
        color=[r['ensemble_score'] for r in recs],
        colorscale='Viridis', showscale=True,
        colorbar=dict(title='Score')
    ),
    text=[f"{r['ensemble_score']:.3f}" for r in recs],
    textposition='outside'
))
fig.update_layout(
    title=' Top 10 Job Recommendations — Ensemble Score',
    xaxis_title='Ensemble Score', yaxis_title='Job Title',
    height=500, template='plotly_white'
)
fig.show()

In [32]:
# Chart 2: Model Score Breakdown (top 5 jobs)

top5   = recs[:5]
models = ['tfidf', 'word2vec', 'knn', 'jaccard', 'recruitment', 'baseline']
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3']

fig2 = go.Figure()
for model, color in zip(models, colors):
    fig2.add_trace(go.Bar(
        name=model.upper(),
        x=[r['title'][:25] for r in top5],
        y=[r['scores'][model] for r in top5],
        marker_color=color
    ))
fig2.update_layout(
    title=' Model-by-Model Score Breakdown (Top 5 Jobs)',
    barmode='group', xaxis_title='Job', yaxis_title='Score',
    height=450, template='plotly_white', legend_title='Model'
)
fig2.show()

In [33]:
# Chart 3: Skill Priority & Salary Impact

fig3 = px.scatter(
    pd.DataFrame(skill_recs),
    x='priority', y='salary_impact',
    text='skill', color='category', size='salary_impact',
    title=' Skill Recommendations: Priority vs Salary Impact',
    labels={'priority': 'Priority Score (0–100)', 'salary_impact': 'Salary Impact ($)'},
    height=500, template='plotly_white'
)
fig3.update_traces(textposition='top center')
fig3.show()

In [34]:
# Chart 4: Salary Prediction with Confidence Interval

sal   = results['metrics']['salary_info']
pred  = sal['predicted']
lo, hi = sal['lower_bound'], sal['upper_bound']

fig4 = go.Figure()
fig4.add_trace(go.Indicator(
    mode="number+delta+gauge",
    value=pred,
    delta={'reference': 50000, 'valueformat': ',.0f', 'prefix': '$'},
    number={'prefix': '$', 'valueformat': ',.0f'},
    gauge={
        'axis': {'range': [30000, 200000]},
        'bar': {'color': '#636EFA'},
        'steps': [
            {'range': [30000, 60000], 'color': '#EF553B'},
            {'range': [60000, 100000], 'color': '#FFA15A'},
            {'range': [100000, 200000], 'color': '#00CC96'}
        ],
        'threshold': {'line': {'color': 'red', 'width': 4}, 'thickness': 0.75, 'value': pred}
    },
    title={'text': f' Predicted Salary<br><sub>Range: ${lo:,.0f} – ${hi:,.0f}</sub>'}
))
fig4.update_layout(height=350, template='plotly_white')
fig4.show()

In [35]:
# Chart 5: Evaluation Metrics Radar

m      = results['metrics']
cats   = ['Precision@K', 'Recall@K', 'F1-Score', 'Wilcoxon (norm)', "Cohen's D (norm)"]
vals   = [
    m['precision_at_k'],
    m['recall_at_k'],
    m['f1_score'],
    max(0, 1 - m['wilcoxon_p']),        # High score = significant
    min(abs(m['cohens_d']) / 2, 1.0)   # Normalised
]

fig5 = go.Figure(go.Scatterpolar(
    r=vals + [vals[0]], theta=cats + [cats[0]],
    fill='toself', name='Metrics',
    line_color='#636EFA'
))
fig5.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title=' Evaluation Metrics Radar',
    height=450, template='plotly_white'
)
fig5.show()

print("\n ALL VISUALIZATIONS COMPLETE!")


🎉 ALL VISUALIZATIONS COMPLETE!


---

## [Done] Pipeline Complete

| Step | What ran |
|---|---|
| 1 | Dependencies installed |
| 2 | Imports & Config |
| 3 | Skill Database (13 skills, real course links) |
| 4 | Data models (CVProfile, JobPosting, RecommendationResult) |
| 5 | TextPreprocessor + CVParser (PDF/DOCX/TXT) |
| 6 | 7 ML models (Baseline, TF-IDF, Word2Vec, Jaccard, KNN, Salary, 6-Stage) |
| 7 | Evaluation metrics (Precision@K, Recall@K, F1, Wilcoxon, Cohen's D) |
| 8 | CSV Data Loader (3 files + demo fallback) |
| 9 | Ensemble Recommendation Engine |
| 10 | Report Generator |
| 11 | Full pipeline execution |
| 12 | Interactive Plotly visualizations |

** Aligned with SDG-8: Decent Work and Economic Growth**
